In [ ]:
!pip install -q --upgrade --force-reinstall numpy
!pip install -q --upgrade --force-reinstall --no-deps rdkit
!pip install -q --upgrade pandas scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.2.2 which is incompatible.
libpysal 4.14.1 requires scikit-learn>=1.4.0, but you have scikit-learn 1.2.2 which is incompatible.
spopt 0.7.0 requires scikit-learn>=1.4.0, but you have scikit-learn 1.2.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.2.2 which is incompatible.
esda 2.9.0 requires scikit-learn>=1.4, but you have scikit-learn 1.2.2 which is incompatible.
mapclassify 2.10.0 requires scikit-learn>=1.4, but you have scikit-learn 1.2.2 which is incompatible.
imbalanced-learn 0.14.2 requires scikit-lear

In [ ]:
import numpy as np
print("numpy:", np.__version__)
import rdkit
print("RDKit OK:", rdkit.__version__)
import pandas as pd
print("pandas OK:", pd.__version__)

numpy: 2.5.2
RDKit OK: 2026.03.5
pandas OK: 3.0.5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/herg_hackathon'

Mounted at /content/drive


In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

df = pd.read_csv(f"{SAVE_DIR}/herg_clean.csv")
print(f"Loaded {len(df)} compounds for splitting.")

def get_scaffold(smiles):
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(smiles), includeChirality=False)
    except Exception:
        return None

df["scaffold"] = df["smiles"].apply(get_scaffold)
df = df.dropna(subset=["scaffold"])
print(f"Compounds with valid scaffold: {len(df)}")

scaffold_groups = df.groupby("scaffold")["compound_id"].apply(list).to_dict()
scaffolds = list(scaffold_groups.keys())

np.random.seed(42)
np.random.shuffle(scaffolds)

train_scaffolds, test_scaffolds = train_test_split(scaffolds, test_size=0.2, random_state=42)

train_ids = set()
for s in train_scaffolds:
    train_ids.update(scaffold_groups[s])
test_ids = set()
for s in test_scaffolds:
    test_ids.update(scaffold_groups[s])

df["split"] = df["compound_id"].apply(lambda x: "train" if x in train_ids else "test")

print(f"Train: {(df['split']=='train').sum()} compounds")
print(f"Test:  {(df['split']=='test').sum()} compounds")
print(f"Train blocker ratio: {df[df['split']=='train']['label'].mean():.3f}")
print(f"Test blocker ratio:  {df[df['split']=='test']['label'].mean():.3f}")

df.to_csv(f"{SAVE_DIR}/herg_split.csv", index=False)
print(f"Saved to {SAVE_DIR}/herg_split.csv")

Loaded 16159 compounds for splitting.
Compounds with valid scaffold: 16159
Train: 12866 compounds
Test:  3293 compounds
Train blocker ratio: 0.554
Test blocker ratio:  0.570
Saved to /content/drive/MyDrive/herg_hackathon/herg_split.csv


In [ ]:
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import numpy as np
import pandas as pd

df = pd.read_csv(f"{SAVE_DIR}/herg_split.csv")
print(f"Featurizing {len(df)} compounds...")

def get_ecfp4(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
    return np.array(fp)

def get_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "HeavyAtoms": Descriptors.HeavyAtomCount(mol),
    }

print("Generating ECFP4 fingerprints...")
fps = df["smiles"].apply(get_ecfp4)
valid_mask = fps.notna()
df = df[valid_mask].reset_index(drop=True)
fps = fps[valid_mask].reset_index(drop=True)
fp_matrix = np.vstack(fps.values)
print(f"Fingerprint matrix shape: {fp_matrix.shape}")

print("Generating RDKit descriptors...")
desc_list = df["smiles"].apply(get_descriptors)
desc_df = pd.DataFrame(list(desc_list))
print(f"Descriptor matrix shape: {desc_df.shape}")

np.save(f"{SAVE_DIR}/fp_matrix.npy", fp_matrix)
desc_df.to_csv(f"{SAVE_DIR}/descriptors.csv", index=False)
df.to_csv(f"{SAVE_DIR}/herg_featurized_meta.csv", index=False)

print("Saved: fp_matrix.npy, descriptors.csv, herg_featurized_meta.csv")
print(f"\nFinal featurized dataset: {len(df)} compounds")
print(f"Train: {(df['split']=='train').sum()}, Test: {(df['split']=='test').sum()}")

Featurizing 16159 compounds...
Generating ECFP4 fingerprints...


Streaming output truncated to the last 5000 lines.
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:14:13] DEPRECATION WARNING: please use MorganGenerator
[10:1

Fingerprint matrix shape: (16159, 2048)
Generating RDKit descriptors...
Descriptor matrix shape: (16159, 8)
Saved: fp_matrix.npy, descriptors.csv, herg_featurized_meta.csv

Final featurized dataset: 16159 compounds
Train: 12866, Test: 3293


In [1]:
!pip install -q --force-reinstall "numpy==1.26.4"
!pip install -q --force-reinstall --no-deps rdkit
!pip install -q "scikit-learn==1.2.2" pandas joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatib

In [1]:
import numpy as np
print("numpy:", np.__version__)
import pandas as pd
print("pandas OK:", pd.__version__)
import rdkit
print("RDKit OK:", rdkit.__version__)

numpy: 1.26.4
pandas OK: 2.2.2
RDKit OK: 2026.03.5


In [2]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/herg_hackathon'

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score)

# Load the same clean dataset (before scaffold split was applied)
df = pd.read_csv(f"{SAVE_DIR}/herg_clean.csv")
fp_matrix = np.load(f"{SAVE_DIR}/fp_matrix.npy")
scaffold_meta = pd.read_csv(f"{SAVE_DIR}/herg_featurized_meta.csv")  # has same compound order as fp_matrix
desc = pd.read_csv(f"{SAVE_DIR}/descriptors.csv")

X = np.hstack([fp_matrix, desc.values])
y = scaffold_meta["label"].values

# --- RANDOM SPLIT (for comparison only) ---
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_train_rand, X_test_rand = X[train_idx], X[test_idx]
y_train_rand, y_test_rand = y[train_idx], y[test_idx]

print(f"Random split - Train: {X_train_rand.shape}, Test: {X_test_rand.shape}")

print("Training Random Forest on RANDOM split...")
rf_random = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
rf_random.fit(X_train_rand, y_train_rand)

probs_rand = rf_random.predict_proba(X_test_rand)[:, 1]
preds_rand = rf_random.predict(X_test_rand)

print("\nRandom Forest — RANDOM SPLIT")
print(f"  Accuracy:  {accuracy_score(y_test_rand, preds_rand):.4f}")
print(f"  Precision: {precision_score(y_test_rand, preds_rand):.4f}")
print(f"  Recall:    {recall_score(y_test_rand, preds_rand):.4f}")
print(f"  F1:        {f1_score(y_test_rand, preds_rand):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_rand, probs_rand):.4f}")
print(f"  PR-AUC:    {average_precision_score(y_test_rand, probs_rand):.4f}")

print("\n" + "="*50)
print("COMPARISON: Scaffold Split vs Random Split (Random Forest)")
print("="*50)
print("Scaffold split (harder, honest):  Accuracy=0.777, ROC-AUC=0.865")
print(f"Random split (easier, inflated):  Accuracy={accuracy_score(y_test_rand, preds_rand):.3f}, ROC-AUC={roc_auc_score(y_test_rand, probs_rand):.3f}")

Random split - Train: (12927, 2056), Test: (3232, 2056)
Training Random Forest on RANDOM split...

Random Forest — RANDOM SPLIT
  Accuracy:  0.8144
  Precision: 0.8212
  Recall:    0.8522
  F1:        0.8364
  ROC-AUC:   0.9001
  PR-AUC:    0.9176

COMPARISON: Scaffold Split vs Random Split (Random Forest)
Scaffold split (harder, honest):  Accuracy=0.777, ROC-AUC=0.865
Random split (easier, inflated):  Accuracy=0.814, ROC-AUC=0.900
